# HW4 - Can We Trust AI in Law and Healthcare? (Base Model vs RAG)

**Goal of this assignment:**  
In this notebook, you will explore how an AI model behaves when answering **law** or **healthcare** questions, both:

- as a **base model** (just the model), and  
- as a **RAG model** (Retrieval-Augmented Generation) that can look up information from a small library of documents.

**Submit**
- PDF export of this notebook (File → Print → Save as PDF)
- The file you add to your RAG database.

**Important notes**
- You must run all code cells in your notebook as you proceed through the assignment. You do this by clicking the play (▶️) button to the left of the cell.
- A pencil icon (✏️) means you have something to write for that step.




---



## Step A - First off, what is RAG?

Watch these two 60 second YouTube shorts for a brief explanation of RAG:

- https://www.youtube.com/shorts/hCWPxZ8wUkY
- https://youtube.com/shorts/qxr7ovnDBHg?si=4HpCIPlD6CeoMME8

### ✏️ Reflection

1. In a few sentences, how would you explain the overall idea of RAG to a friend?
2. Why might RAG reduce hallucinations?
3. How would RAG be useful (or not useful) for your creative chatbot from Homework 3?

[YOUR NOTES HERE]



---



## ✏️ Step B - Decide your domain

In the code cell below, you will choose between one of law or healthcare for this assignment. Run the code cell upon selection.

In [ ]:
# You only need to work with ONE domain for this homework.

DOMAIN = ""  # ✏️ TODO: Right now, this is empty: "". Set to "law" or "healthcare"

assert DOMAIN in ["law", "healthcare"], "DOMAIN must be 'law' or 'healthcare'"
print("✅ You have chosen the domain:", DOMAIN)
ROLE_TEXT = {
    "law":  ("law", "lawyer", "legal advice", "specific legal statutes"),
    "healthcare": ("healthcare", "doctor", "medical advice", "specific diagnoses or step-by-step self-treatment plans"),
}
topic, profession, advice, examples = ROLE_TEXT[DOMAIN]




---



## Step C - Create your document database

RAG (Retrieval-Augmented Generation) depends on having a database of real documents that the model can use to find factual information.

1. First, on your computer, unzip the `docs.zip` file provided for this assignment. It will give you a `docs` folder.
2. **IMPORTANT**: Delete the `docs.zip` file. after you unzip it.


### ✏️ Reflection

- Look at the four documents you're using for your selected domain (from Step B). What is the source for each (four total)?
- Find an additional source you can add. Why would you add that source?

[YOUR NOTES HERE]

### Update the documents

1. Now, **add the source** identified in your Reflection to your `docs/` folder.
    - Format the document such that it begins with the Source, just like the other documents.
    - Name it either `law_5.txt` or `healthcare_5.txt`.

2. Zip the updated `docs` folder. This should give you a file named `docs.zip`. If it does not, rename the file accordingly.


## Step D - Set up your document database in Google Drive

In this step, we'll connect your Google Drive and extract the documents needed for this assignment.

1. **Upload** the `docs.zip` file to your Google Drive.

2. **Run the code cell below**. It will:  
   - Mount your Google Drive,  
   - Search for the ZIP file,  
   - Unzip it into a folder named `/content/docs`, and  
   - Print the list of files found.

> 💡 If you don’t see the file, double-check its exact location in Drive and update the path in the cell accordingly.

In [ ]:
from google.colab import drive
import os, zipfile, glob

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Try to locate docs.zip anywhere in Drive
search_pattern = '/content/drive/My Drive/**/docs.zip'
matches = glob.glob(search_pattern, recursive=True)

if not matches:
    print("⚠️ Could not find docs.zip in your Drive.")
    print("Please upload it manually to your Google Drive (e.g., My Drive/AI_in_Organizations_HW4/).")
else:
    drive_zip_path = matches[0]
    print(f"✅ Found ZIP file at: {drive_zip_path}")

    # 3. Unzip to /content/docs
    os.makedirs('/content/docs', exist_ok=True)
    with zipfile.ZipFile(drive_zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/docs')

    print("✅ Extracted files to /content/docs")




---



## Step E - Set up your API key via Colab Secrets

1. In Colab, open the **left sidebar** and click the **key icon (🔑 Secrets)**.  
2. Click **Add new secret**.  
3. Set **Name** exactly to: `OPENAI_API_KEY`  
4. Paste your key from Homework 3 into **Value**. Make sure **Notebook Access** is enabled. If you cannot find your key, please create another key following the same process as in Homework 3.
5. Run the two cells below (hit the ▶️ Play button). You should see a **Success** message for both.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') or ""
if not os.environ["OPENAI_API_KEY"].startswith("sk-"):
    raise ValueError(
        "OPENAI_API_KEY not found. In the left sidebar (🔑 Secrets), add OPENAI_API_KEY, then re-run."
    )
print("✅ Success! API key detected in Secrets.")

In [ ]:
!pip install --quiet openai numpy

from openai import OpenAI
import numpy as np
import glob
import os
import textwrap

# The OpenAI client expects your API key in the environment as OPENAI_API_KEY.
# Make sure this is set in your environment / notebook settings.
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("✅ Success! Libraries imported and OpenAI client initialized.")



---



## ✏️ Step F - Prepare the queries for testing

Look at the code cell below to see the queries we are using to test our base and RAG models. Run this cell. These queries will test whether the RAG model will reference the appropriate documents.

In [ ]:
law_questions = [
    "My landlord hasn’t fixed a serious mold problem in my apartment for three months. What legal options do I have?",
    "Can my university search my dorm room without my permission?",
    "If I am arrested at a protest, what should I do first?",
    "Can I use legal information I find on social media as my main source when making decisions about my case?"
]

healthcare_questions = [
    "I’ve had bad chest pain on and off for the last 6 hours. What should I do?",
    "Can I stop my antibiotics once I feel better, even if there are pills left?",
    "Is it safe to take my friend’s prescription anxiety medication before an exam?",
    "Why do I feel like my antibiotics aren't working as well? I've been using them every time I feel a bit sick."
]

Think about a query you can use to test whether the document you just added to your RAG database is referenced.

In the code cell below, write this query down. Run the cell.

In [ ]:
MY_QUERY = "" # TODO: ✏️ Fill this in

Run the cell below to add your query to the set of queries that will be used to test the mode.

In [ ]:
if DOMAIN == "law":
    questions = law_questions + [MY_QUERY]
else:
    questions = healthcare_questions + [MY_QUERY]

print(f"✅ Loaded {len(questions)} questions for domain: {DOMAIN}\n")
for i, q in enumerate(questions, start=1):
    print(f"Q{i}: {q}")

## 🛑 STOP: Did you run all code cells above?

---

## Step G - Prepare the documents for RAG

Now that we have our database of documents, we need a way for the AI model to decide which ones are most relevant to a question.

### Step G.1 - Load the documents

Each document lives in your docs/ folder (for example, `law_1.txt`, `healthcare_2.txt`, etc.). The function below reads these files and stores them in a list so that the model can later search through them. Run the cell below.

In [ ]:
def load_domain_docs(domain: str):
    """
    Load text files for a domain from the docs/ folder.
    Expected filenames:
      - docs/law_1.txt, docs/law_2.txt, ...
      - docs/healthcare_1.txt, docs/healthcare_2.txt, ...
    """
    pattern = f"*/docs/{domain}_*.txt"
    filepaths = sorted(glob.glob(pattern))

    docs = []
    for path in filepaths:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read().strip()
        doc_id = os.path.splitext(os.path.basename(path))[0]  # e.g., "law_1"
        docs.append({
            "id": doc_id,
            "title": doc_id.replace("_", " ").title(),
            "text": text
        })
    return docs

docs = load_domain_docs(DOMAIN)

if len(docs) == 5:
  print(f"✅ Loaded {len(docs)} documents for domain: {DOMAIN}\n")
  for d in docs:
      print("-", d["id"])
else:
  print(f"⚠️ Expected 5 documents for domain: {DOMAIN}")

### Step G.2 - Convert text to numbers (embeddings)

Right now, our documents are just plain text. To compare them mathematically, we'll turn each one into a vector, which is just a list of numbers that abstractly captures the meaning of the text. Vectors might look like this:

`[0.5, 0.3, 0.2, ...]`

Two documents with similar meanings will have similar vectors. Run the next cell to generate these embeddings automatically using OpenAI's embedding model:

In [ ]:
def get_embedding(text: str, model: str = "text-embedding-3-small"):
    """Get an embedding vector for the given text using OpenAI embeddings."""
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return np.array(response.data[0].embedding)


# Pre-compute embeddings for all docs in the chosen domain
for doc in docs:
    doc["embedding"] = get_embedding(doc["text"])

print("✅ Computed embeddings for", len(docs), "documents.")

### Step G.3 - Measure similarity between documents

Once each document is represented as a vector, we can measure how close two vectors are in meaning. We'll use a method called cosine similarity, which gives a number between -1 and 1. -1 means “very different,” and +1 means “almost identical.”

Run the cell below to define the similarity function:

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

Then, run the cell below. This cell uses the similarity function to find the top k most similar documents. For example, if `k = 2`, then it finds the two most similar.

In [ ]:
def retrieve_docs(query: str, k: int = 2):
    """Retrieve the top-k most similar documents to the query within the chosen domain."""
    query_emb = get_embedding(query)

    scored = []
    for doc in docs:
        sim = cosine_similarity(query_emb, doc["embedding"])
        scored.append((sim, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    top_docs = [doc for sim, doc in scored[:k]]
    return top_docs

🧠 **What's Happening Conceptually?**

- Each document is converted into a vector of numbers
- When a user asks a question, that question is also turned into a vector
- The model finds the documents whose vectors are closest to the question’s vector
- Those documents are passed into the prompt so the model can answer using real information

### ✏️ Reflection

1. What do you think embeddings allow the model to do that plain text cannot?
2. Do you think two texts can have similar vectors but still mean different things in context? Give an example or hypothesis.

[YOUR NOTES HERE]



---



## Step H - Create the models

Now that we've prepared our documents and learned how to measure which ones are most relevant, it's time to connect everything together by defining two models that can answer questions.

We'll use:

1. **Base model**: a simple version of the AI that only uses its internal knowledge to answer questions. This is like asking ChatGPT directly without providing any context.
2. **RAG (Retrieval-Augmented Generation) model**: a more advanced version that uses the same base model, but also retrieves the most relevant documents first and then bases its answer on them. This helps it stay closer to verified information and reduces hallucinations.

Run the two cells below to initiate both.


In [ ]:
# @title Define base model helper (no retrieval)

def ask_base_model(question: str) -> str:
    """Ask the base model without retrieval."""

    system_prompt = f"""
    You are an AI assistant answering questions about {topic}.
    You are NOT a {profession}. You must:
    - Be honest about what you do and do not know.
    - Avoid giving definitive {advice}.
    - Encourage the user to consult qualified professionals for serious or urgent issues.
    - Avoid making up {examples}.
    """.strip()

    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )
    return response.output_text

In [ ]:
# @title Define RAG model helper (with retrieval)

def ask_rag_model(question: str, k: int = 3):
    """Ask the model with Retrieval-Augmented Generation."""
    retrieved = retrieve_docs(question, k=k)

    # Build context text from retrieved docs
    context_parts = []
    for doc in retrieved:
        context_parts.append(f"Title: {doc['title']}\nContent:\n{doc['text']}")
    context_text = "\n\n---\n\n".join(context_parts)

    system_prompt = f"""
    You are an AI assistant answering questions about {topic}.
    You are NOT a {profession}. You must:
    - Be honest about what you do and do not know.
    - Avoid giving definitive {advice}.
    - Encourage the user to consult qualified professionals for serious or urgent issues.
    - Avoid making up {examples}.
    """.strip()

    user_message = (
        f"User question:\n{question}\n\n"
        f"Here are reference passages you may use:\n{context_text}\n\n"
        "Using only the information in these passages (plus general common-sense reasoning, "
        "but no invented facts), answer the user's question. If the information is missing "
        "or incomplete, say that explicitly."
    )

    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    answer = response.output_text
    return answer, retrieved

### ✏️ Reflection

In both functions, the input variable defines what gets sent to the AI model.
Let's look carefully at how it differs between the base model and the RAG model.

Look at the `input` variable within the `ask_base_model` function (in the first cell). It can be found here:

```
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )
```

Then, look at the `input` variable within the `ask_rag_model` function (in the second cell). It can be found here:

```
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
```

You will notice that the `system_prompt`, used in the `system` role, is the same for both. However, the content for the `user` role is not.

1. What is the difference between the content of `question` (in the first `user` role) and `user_message` (in the second `user` role)?
- *Hint: Look at the variables in the code cells.*
2. Why does the content change for the user role in the RAG version?

[YOUR NOTES HERE]



---



## Step I - Explore the base model (no retrieval)

Run the cell below to use the base model on the queries we defined in Step E.

In [ ]:
# @title Run the base model on all questions

base_answers = []

for i, q in enumerate(questions, start=1):
    print(f"==== Base Model Answer for Question {i} ====")
    print("Question:", q)

    answer = ask_base_model(q)
    base_answers.append(answer)

    print("\nAnswer:\n")
    print(textwrap.fill(answer, width=100))
    print("\n" + "=" * 100 + "\n")

### ✏️ Reflection

For **two** of the questions above, write 1-3 sentences each:

- What seems **helpful** about the base model's answer?
- What, if anything, **worries you** about someone acting on this information in real life?

[YOUR NOTES HERE]



---



## Step J - Explore the RAG model (with retrieval)

Now we will ask the **RAG model** the **same questions** as in Step H. Remember that the RAG model is allowed to use the short documents you loaded from the `docs/` folder.

Run the cell below.

In [ ]:
# @title Run the RAG model on all questions

rag_answers = []
rag_retrieved = []

for i, q in enumerate(questions, start=1):
    print(f"==== RAG Model Answer for Question {i} ====")
    print("Question:", q)
    answer, retrieved_docs = ask_rag_model(q, k=2)
    rag_answers.append(answer)
    rag_retrieved.append(retrieved_docs)

    print("\nAnswer:\n")
    print(textwrap.fill(answer, width=100))
    print("\nRetrieved documents used:")
    for doc in retrieved_docs:
        print(" -", doc["id"], "(", doc["text"][:doc["text"].find("\n")], ")")
    print("\n" + "="*100 + "\n")

### ✏️ Reflection

For **two** of the queries you ran above:

For each, compare the **base** and **RAG** answers:

- How is the RAG answer different from the base answer?
- Does the RAG answer feel **more accurate or grounded**? Why or why not?
- Is there **anything still missing or potentially risky** in the RAG answer?
- Based on this, what do you think the **limits** of RAG are in your chosen domain?

[YOUR NOTES HERE]



---



## Step K - Conceptual reflection

Answer the following questions in your own words. Be sure to answer all questions.

### ✏️ 1. Why does RAG matter more in law / healthcare than in some other domains?

In 3–5 sentences, explain why RAG is especially important in your chosen domain (law or healthcare). Refer to at least **one specific example** from your experiments.

[YOUR NOTES HERE]

### ✏️ 2. Imagining Fine-Tuning (No Coding)

Watch this YouTube Short about fine-tuning:

- https://www.youtube.com/shorts/0o1qKoPRrRs?feature=share

After watching this video:

- Explain in a few sentences what fine-tuning is. How does it differ from RAG?

Even though you did not fine-tune a model in this assignment:

- How might fine-tuning improve the model compared to just using RAG or a system prompt?
- If you could fine-tune this model for your chosen domain (law or healthcare),
what specific behaviors would you want it to learn?
(For example, being cautious, asking clarifying questions, or flagging when it's uncertain.)
- What potential risks might come with fine-tuning in sensitive areas like law or healthcare?

[YOUR NOTES HERE]

### ✏️ 3. Should organizations deploy systems like this?

Reflect on the following:

- Should an organization deploy a model like this for public use or learning?
  - What potential benefits do you see (e.g., accessibility, faster guidance, reduced workload for staff)?
  - What potential risks concern you (e.g., misinformation, overreliance, loss of human judgment)?
- What technological or policy improvements would need to happen before deployment becomes safe enough?

[YOUR NOTES HERE]



---



## Submission

Make sure you've run all code cells, and make sure you've completed all writing requirements. Once you've done that, proceed with submission by obtaining the notebook PDF (File → Print → Save as PDF) and the file you added to the RAG database. Submit both on Canvas.